In [ ]:
import sys
import cv2
import torch
import numpy as np
from pathlib import Path
from skimage.restoration import denoise_tv_chambolle


from Models.DeepLabV3Plus.modeling import deeplabv3plus_resnet101

RAW_DIR = Path(r"/home/khoa/Workspace/CardioVis/Backend-Inference/exports/export-4-10/extracted/images")
OUTPUT_VIDEO = Path(r"/home/khoa/Workspace/CardioVis/Backend-Inference/exports/export-4-10/guideline_overlay.mp4")
# CHECKPOINT = "checkpoints/cardio/cardio_run/fold1/best_fea2.pth"
IMG_SIZE = 224
MAX_FRAMES = 1000
FPS = 18.0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------------------------------------------------------------------
# Developer config — edit only this block when switching features/checkpoints.
# Class index i must match logit i from the model head. If masks look empty,
# check len(CLASS_NAMES) == NUM_MODEL_CLASSES after the model cell, or set
# MODEL_CLASS_TO_SEMANTIC to a uint8 LUT of length NUM_MODEL_CLASSES.
#
# label_fea3_p10 (+ legacy 1–3): TAG_TO_CLASS maps tag string -> model id (0 = background).
# ---------------------------------------------------------------------------
MODEL_CLASS_TO_SEMANTIC = None  # or np.array([...], dtype=np.uint8) length NUM_MODEL_CLASSES

TAG_TO_CLASS = {
    "epicardial adipose tissue": 1,
    "pericardium": 2,
    "phrenic nerve": 3,
    "aortic root": 4,
    "auricles": 5,
    "epicardial fat on aortic": 6,
    "grasper": 7,
    "needle holders": 8,
}
# CLASS_NAMES[i] must match logit i (9 classes: 0..8).
CLASS_NAMES = [
    "Background",
    "Epicardial adipose tissue",
    "Pericardium",
    "Phrenic nerve",
    "Aortic root",
    "Auricles",
    "Epicardial fat on aortic",
    "Grasper",
    "Needle holders",
]
ALL_FOREGROUND_IDS = tuple(range(1, len(CLASS_NAMES)))
CLASS_COLORS_BGR = [
    (0, 0, 0),  # 0 Background
    (0, 140, 255),  # 1 Epicardial adipose tissue
    (0, 255, 0),  # 2 Pericardium
    (0, 0, 255),  # 3 Phrenic nerve
    (0, 255, 100),  # 4 Aortic root
    (255, 80, 200),  # 5 Auricles
    (40, 40, 180),  # 6 Epicardial fat on aortic
    (128, 128, 128),  # 7 Grasper
    (0, 200, 255),  # 8 Needle holders
]
GUIDELINE_ANCHOR_CLASS_ID = TAG_TO_CLASS["pericardium"]  # 2 — centerline / band
OVERLAY_ALPHA_CLASS_IDS = tuple(c for c in ALL_FOREGROUND_IDS if c != GUIDELINE_ANCHOR_CLASS_ID)
CALLOUT_CLASS_IDS = ALL_FOREGROUND_IDS
DISPLAY_CLASS_NAMES = {}  # optional {class_id: "short label"}
BLINK_WARNING_CLASS_ID = TAG_TO_CLASS["phrenic nerve"]  # 3 — or None
BLINK_WARNING_PERIOD_FRAMES = 8

EXPORT_MASK_LAYERS = [
    {
        "class_id": 1,
        "subdir": "class01_epicardial_adipose",
        "file_prefix": "class01_",
        "json_key": "class_1_epicardial_adipose_tissue",
        "note": "0/255 binary, model id 1 (epicardial adipose tissue).",
    },
    {
        "class_id": 2,
        "subdir": "class02_pericardium",
        "file_prefix": "class02_",
        "json_key": "class_2_pericardium_full_segmentation",
        "note": "0/255 binary, model id 2 (pericardium); guideline anchor.",
    },
    {
        "class_id": 3,
        "subdir": "class03_phrenic",
        "file_prefix": "class03_",
        "json_key": "class_3_phrenic_nerve",
        "note": "0/255 binary, model id 3 (phrenic nerve).",
    },
    {
        "class_id": 4,
        "subdir": "class04_aortic_root",
        "file_prefix": "class04_",
        "json_key": "class_4_aortic_root",
        "note": "0/255 binary, model id 4 (aortic root).",
    },
    {
        "class_id": 5,
        "subdir": "class05_auricles",
        "file_prefix": "class05_",
        "json_key": "class_5_auricles",
        "note": "0/255 binary, model id 5 (auricles).",
    },
    {
        "class_id": 6,
        "subdir": "class06_epicardial_fat_aortic",
        "file_prefix": "class06_",
        "json_key": "class_6_epicardial_fat_on_aortic",
        "note": "0/255 binary, model id 6 (epicardial fat on aortic).",
    },
    {
        "class_id": 7,
        "subdir": "class07_grasper",
        "file_prefix": "class07_",
        "json_key": "class_7_grasper",
        "note": "0/255 binary, model id 7 (grasper).",
    },
    {
        "class_id": 8,
        "subdir": "class08_needle_holders",
        "file_prefix": "class08_",
        "json_key": "class_8_needle_holders",
        "note": "0/255 binary, model id 8 (needle holders).",
    },
]

# Downstream JSON keeps key "pericardium_class"; value is the guideline anchor id.
PERICARDIUM_CLASS = GUIDELINE_ANCHOR_CLASS_ID

ALPHA = 0.5
GUIDELINE_COLOR_BGR = (0, 255, 0)
OFFSET_CM = 1.2
PIXELS_PER_CM = 20.0
OFFSET_COLOR_POS_BGR = (0, 140, 255)
OFFSET_COLOR_NEG_BGR = (255, 0, 0)

DASH_LEN = 12
GAP_LEN = 8
LINE_THICKNESS = 2
OFFSET_LINE_THICKNESS = 2

TV_WEIGHT = 0.15
TEMPORAL_ALPHA = 0.25
MAX_CENTERLINE_JUMP_PX = 25

BAND_WIDTH_SCALE = 12.0
BAND_ALPHA = 0.18
BAND_COLOR_BGR = (0, 255, 0)

CALLOUT_TEXT_COLOR = (255, 255, 255)
CALLOUT_BG_COLOR = (20, 20, 20)
CALLOUT_ARROW_COLOR = (255, 255, 255)
CALLOUT_FONT_SCALE = 0.55
CALLOUT_THICKNESS = 1
CALLOUT_PADDING = 6
CALLOUT_MIN_MASK_PIXELS = 1
CALLOUT_OFFSET_CM = 2.0

DANGER_TRIANGLE_COLOR = (0, 0, 255)
DANGER_TRIANGLE_OUTLINE = (255, 255, 255)


In [2]:
def _normalize_state_dict(ckpt):
    if isinstance(ckpt, dict):
        if "state_dict" in ckpt:
            ckpt = ckpt["state_dict"]
        elif "model" in ckpt and isinstance(ckpt["model"], dict):
            ckpt = ckpt["model"]
    out = {}
    for k, v in ckpt.items():
        nk = k[7:] if k.startswith("module.") else k
        out[nk] = v
    return out


def _infer_num_classes(state):
    key = "classifier.classifier.3.weight"
    if key not in state:
        raise KeyError(
            f"Cannot infer num_classes (missing {key!r}). Sample keys: {list(state.keys())[:10]}"
        )
    return int(state[key].shape[0])


raw_ckpt = torch.load(CHECKPOINT, map_location=DEVICE, weights_only=False)
state = _normalize_state_dict(raw_ckpt)
NUM_MODEL_CLASSES = _infer_num_classes(state)
# #region agent log
import json as _agent_json, time as _agent_time
_AGENT_LOG = r"/home/khoa/Workspace/CardioVis/Backend-Inference/.cursor/debug-35efb6.log"
def _agent_log(hid, loc, msg, data):
    with open(_AGENT_LOG, "a") as _f:
        _f.write(_agent_json.dumps({"sessionId": "35efb6", "hypothesisId": hid, "location": loc, "message": msg, "data": data, "timestamp": int(_agent_time.time() * 1000)}) + "\n")
_agent_log("H2", "nb:after_infer", "checkpoint_head", {"NUM_MODEL_CLASSES": NUM_MODEL_CLASSES, "weight_out_c": int(state["classifier.classifier.3.weight"].shape[0])})
# #endregion

model = deeplabv3plus_resnet101(
    num_classes=NUM_MODEL_CLASSES, output_stride=8, pretrained_backbone=False
)
model.load_state_dict(state, strict=True)
# #region agent log
_agent_log("H2", "nb:after_load_state_dict", "strict_load_ok", {"NUM_MODEL_CLASSES": NUM_MODEL_CLASSES})
# #endregion
model = model.to(DEVICE)
model.eval()


def model_pred_to_semantic(pred):
    """Map argmax class ids to uint8 semantic ids (same indexing as CLASS_NAMES unless using LUT)."""
    p = np.asarray(pred, dtype=np.int32)
    if MODEL_CLASS_TO_SEMANTIC is not None:
        lut = np.asarray(MODEL_CLASS_TO_SEMANTIC, dtype=np.uint8)
        if lut.shape[0] != NUM_MODEL_CLASSES:
            raise ValueError(
                f"MODEL_CLASS_TO_SEMANTIC length {lut.shape[0]} != NUM_MODEL_CLASSES {NUM_MODEL_CLASSES}"
            )
        return lut[p]
    if len(CLASS_NAMES) == NUM_MODEL_CLASSES:
        return p.astype(np.uint8)
    raise ValueError(
        f"NUM_MODEL_CLASSES={NUM_MODEL_CLASSES} but len(CLASS_NAMES)={len(CLASS_NAMES)}. "
        "Set CLASS_NAMES to match checkpoint order/size, or set MODEL_CLASS_TO_SEMANTIC to a "
        "uint8 ndarray of shape (NUM_MODEL_CLASSES,) mapping model class ids to semantic ids."
    )


print(f"Model loaded (num_classes={NUM_MODEL_CLASSES}).")
# #region agent log
_agent_log("H3", "nb:before_assert", "class_names_vs_num_model", {"len_CLASS_NAMES": len(CLASS_NAMES), "NUM_MODEL_CLASSES": NUM_MODEL_CLASSES})
# #endregion
if MODEL_CLASS_TO_SEMANTIC is None:
    assert len(CLASS_NAMES) == NUM_MODEL_CLASSES, (
        f"len(CLASS_NAMES)={len(CLASS_NAMES)} != NUM_MODEL_CLASSES={NUM_MODEL_CLASSES}; "
        "align CLASS_NAMES with the checkpoint or set MODEL_CLASS_TO_SEMANTIC."
    )


Model loaded (num_classes=11).


AssertionError: len(CLASS_NAMES)=9 != NUM_MODEL_CLASSES=11; align CLASS_NAMES with the checkpoint or set MODEL_CLASS_TO_SEMANTIC.

In [ ]:
def preprocess_frame(frame):
    h, w = frame.shape[:2]
    img = cv2.resize(frame, (IMG_SIZE, IMG_SIZE))
    img = np.clip(img, 0, 255).astype(np.float32) / 255.0
    img = torch.from_numpy(img).float().permute(2, 0, 1).unsqueeze(0).to(DEVICE)
    return img, (h, w)


def draw_dashed_line(img, pt1, pt2, color, thickness, dash_len, gap_len):
    x1, y1 = float(pt1[0]), float(pt1[1])
    x2, y2 = float(pt2[0]), float(pt2[1])
    length = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)
    if length < 1e-6:
        return
    step = dash_len + gap_len
    n = int(length / step) + 1
    for i in range(n):
        t0 = min(i * step / length, 1.0)
        t1 = min((i * step + dash_len) / length, 1.0)
        if t0 >= 1:
            break
        p0 = (int(x1 + t0 * (x2 - x1)), int(y1 + t0 * (y2 - y1)))
        p1 = (int(x1 + t1 * (x2 - x1)), int(y1 + t1 * (y2 - y1)))
        cv2.line(img, p0, p1, color, thickness)


def get_pericardium_centerline(mask, gap_threshold=5, smooth_win=3):
    """Lấy đường centerline trong lòng mask: mỗi cột x lấy y_min, y_max của mask rồi y_mid = (min+max)/2."""
    h, w = mask.shape[:2]
    points_by_x = []
    for x in range(w):
        col = mask[:, x]
        ys = np.where(col > 0)[0]
        if len(ys) > 0:
            y_mid = (ys.min() + ys.max()) / 2.0
            points_by_x.append((x, y_mid))
    if not points_by_x:
        return []
    points_by_x.sort(key=lambda p: p[0])
    if smooth_win >= 3 and len(points_by_x) >= smooth_win:
        xs = np.array([p[0] for p in points_by_x])
        ys = np.array([p[1] for p in points_by_x])
        kernel = np.ones(smooth_win) / smooth_win
        ys_smooth = np.convolve(ys, kernel, mode="same")
        points_by_x = list(zip(xs.tolist(), ys_smooth.tolist()))
    segments = []
    seg = [points_by_x[0]]
    for i in range(1, len(points_by_x)):
        if points_by_x[i][0] - points_by_x[i - 1][0] > gap_threshold:
            if len(seg) >= 2:
                segments.append(seg)
            seg = [points_by_x[i]]
        else:
            seg.append(points_by_x[i])
    if len(seg) >= 2:
        segments.append(seg)
    return segments


def tv_denoise_1d(y, weight):
    """TV-like denoising (L1 on total variation)."""
    if y is None:
        return y
    y = np.asarray(y, dtype=np.float64)
    if y.ndim != 1 or len(y) < 3:
        return y
    return denoise_tv_chambolle(y, weight=weight)



def smooth_centerline_segment(seg_points, prev_seg_points=None):
    """Làm trơn centerline: TV-like theo không gian + EMA theo thời gian."""
    if not seg_points or len(seg_points) < 2:
        return seg_points

    xs = np.array([p[0] for p in seg_points], dtype=np.float64)
    ys = np.array([p[1] for p in seg_points], dtype=np.float64)

    ys_tv = tv_denoise_1d(ys, weight=TV_WEIGHT)

    if prev_seg_points is not None and len(prev_seg_points) >= 2:
        prev_xs = np.array([p[0] for p in prev_seg_points], dtype=np.float64)
        prev_ys = np.array([p[1] for p in prev_seg_points], dtype=np.float64)
        # Interpolate previous line to current x-grid
        prev_ys_interp = np.interp(xs, prev_xs, prev_ys)

        # Clamp sudden y jumps to keep temporal consistency
        dy = ys_tv - prev_ys_interp
        dy = np.clip(dy, -MAX_CENTERLINE_JUMP_PX, MAX_CENTERLINE_JUMP_PX)
        ys_tv = prev_ys_interp + dy

        # EMA blend
        ys_tv = (1.0 - TEMPORAL_ALPHA) * prev_ys_interp + TEMPORAL_ALPHA * ys_tv

    return list(zip(xs.tolist(), ys_tv.tolist()))



def point_inside_mask(mask_2d, x, y):
    xi = int(round(float(x)))
    yi = int(round(float(y)))
    if yi < 0 or xi < 0:
        return False
    if yi >= mask_2d.shape[0] or xi >= mask_2d.shape[1]:
        return False
    return mask_2d[yi, xi] > 0



def project_along_dir_to_inside(mask_2d, center_point, dir_unit, target_offset_px):
    """Project a point back inside the mask along dir_unit."""
    cx, cy = float(center_point[0]), float(center_point[1])
    dx, dy = float(dir_unit[0]), float(dir_unit[1])

    max_d = int(round(float(target_offset_px)))
    if max_d < 0:
        return None

    # If already inside at target offset, keep it.
    x_t = cx + target_offset_px * dx
    y_t = cy + target_offset_px * dy
    if point_inside_mask(mask_2d, x_t, y_t):
        return (x_t, y_t)

    for d in range(max_d, -1, -1):
        x = cx + d * dx
        y = cy + d * dy
        if point_inside_mask(mask_2d, x, y):
            return (x, y)
    return None



def compute_parallel_lines(seg_points, pericardium_mask, offset_px):
    """Tạo 2 polyline song song (2 phía pháp tuyến), đảm bảo nằm trong pericardium mask."""
    if not seg_points or len(seg_points) < 2:
        return [], []

    pts = np.array(seg_points, dtype=np.float64)  # (N,2): x,y
    n = len(pts)

    # Tangents via finite differences => normals
    tangents = np.zeros_like(pts)
    for i in range(n):
        if i == 0:
            t = pts[i + 1] - pts[i]
        elif i == n - 1:
            t = pts[i] - pts[i - 1]
        else:
            t = pts[i + 1] - pts[i - 1]
        norm = np.linalg.norm(t)
        if norm < 1e-8:
            tangents[i] = np.array([1.0, 0.0], dtype=np.float64)
        else:
            tangents[i] = t / norm

    # Normal in image coordinates: (-ty, tx)
    normals = np.stack([-tangents[:, 1], tangents[:, 0]], axis=1)

    pos_points = []
    neg_points = []

    for i in range(n):
        cx, cy = pts[i]
        nx, ny = normals[i]

        # Positive side
        p = project_along_dir_to_inside(
            pericardium_mask,
            (cx, cy),
            (nx, ny),
            offset_px,
        )
        if p is not None:
            pos_points.append(p)

        # Negative side
        p = project_along_dir_to_inside(
            pericardium_mask,
            (cx, cy),
            (-nx, -ny),
            offset_px,
        )
        if p is not None:
            neg_points.append(p)

    return pos_points, neg_points



def compute_parallel_band_polygon(seg_points, pericardium_mask, halfwidth_px):
    """Tạo polygon vùng band giữa 2 biên song song (±halfwidth_px) nằm trong mask."""
    if not seg_points or len(seg_points) < 2:
        return None

    pts = np.array(seg_points, dtype=np.float64)  # (N,2): x,y
    n = len(pts)

    # Tangents via finite differences => normals
    tangents = np.zeros_like(pts)
    for i in range(n):
        if i == 0:
            t = pts[i + 1] - pts[i]
        elif i == n - 1:
            t = pts[i] - pts[i - 1]
        else:
            t = pts[i + 1] - pts[i - 1]
        norm = np.linalg.norm(t)
        if norm < 1e-8:
            tangents[i] = np.array([1.0, 0.0], dtype=np.float64)
        else:
            tangents[i] = t / norm

    # Normal in image coordinates: (-ty, tx)
    normals = np.stack([-tangents[:, 1], tangents[:, 0]], axis=1)

    pos_pts = []
    neg_pts = []

    for i in range(n):
        cx, cy = pts[i]
        nx, ny = normals[i]

        p_pos = project_along_dir_to_inside(
            pericardium_mask,
            (cx, cy),
            (nx, ny),
            halfwidth_px,
        )
        p_neg = project_along_dir_to_inside(
            pericardium_mask,
            (cx, cy),
            (-nx, -ny),
            halfwidth_px,
        )

        if p_pos is None or p_neg is None:
            continue

        pos_pts.append(p_pos)
        neg_pts.append(p_neg)

    if len(pos_pts) < 2 or len(neg_pts) < 2:
        return None

    # Polygon: pos theo thứ tự, neg theo thứ tự đảo để đóng
    poly = pos_pts + neg_pts[::-1]
    return poly



def fill_polygon_alpha(overlay_img, poly_points, color_bgr, alpha):
    """Fill polygon lên overlay với alpha (trộn màu nhẹ)."""
    if poly_points is None or len(poly_points) < 3:
        return

    h, w = overlay_img.shape[:2]
    mask_poly = np.zeros((h, w), dtype=np.uint8)

    pts = np.array([(int(round(x)), int(round(y))) for x, y in poly_points], dtype=np.int32)
    pts = pts.reshape(-1, 1, 2)

    cv2.fillPoly(mask_poly, [pts], 255)

    color = np.array(color_bgr, dtype=np.float32)
    overlay_sel = overlay_img[mask_poly == 255].astype(np.float32)
    overlay_img[mask_poly == 255] = ((1.0 - alpha) * overlay_sel + alpha * color).astype(np.uint8)


def draw_solid_polyline(img, points, color, thickness):
    if not points or len(points) < 2:
        return
    pts = np.array([(int(round(x)), int(round(y))) for x, y in points], dtype=np.int32)
    pts = pts.reshape(-1, 1, 2)
    cv2.polylines(img, [pts], isClosed=False, color=color, thickness=thickness)



def draw_dashed_polyline(img, points, color, thickness, dash_len, gap_len, step_px=2):
    """Vẽ polyline (đường mở) dạng nét đứt."""
    if not points or len(points) < 2:
        return
    pts = np.array(points, dtype=np.float64)
    n = len(pts)
    samples = []
    d_total = 0.0
    for i in range(n - 1):
        p1, p2 = pts[i], pts[i + 1]
        seg_len = np.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)
        if seg_len < 1e-6:
            continue
        num_steps = max(1, int(seg_len / step_px))
        for k in range(num_steps + 1):
            t = k / num_steps if num_steps > 0 else 1
            px = p1[0] + t * (p2[0] - p1[0])
            py = p1[1] + t * (p2[1] - p1[1])
            samples.append(((px, py), d_total))
            if k < num_steps:
                d_total += seg_len / num_steps
    if len(samples) < 2:
        return
    step = dash_len + gap_len
    in_dash = [((s[1] % step) < dash_len) for s in samples]
    for j in range(len(samples) - 1):
        if in_dash[j] and in_dash[j + 1]:
            p0 = (int(samples[j][0][0]), int(samples[j][0][1]))
            p1 = (int(samples[j + 1][0][0]), int(samples[j + 1][0][1]))
            cv2.line(img, p0, p1, color, thickness)


def draw_dashed_contour(img, contour, color, thickness, dash_len, gap_len, step_px=2):
    if contour is None or len(contour) < 2:
        return
    pts = contour.reshape(-1, 2).astype(np.float64)
    n = len(pts)
    samples = []
    d_total = 0.0
    for i in range(n):
        p1 = pts[i]
        p2 = pts[(i + 1) % n]
        seg_len = np.sqrt((p2[0] - p1[0]) ** 2 + (p2[1] - p1[1]) ** 2)
        if seg_len < 1e-6:
            continue
        num_steps = max(1, int(seg_len / step_px))
        for k in range(num_steps + 1):
            t = k / num_steps if num_steps > 0 else 1
            px = p1[0] + t * (p2[0] - p1[0])
            py = p1[1] + t * (p2[1] - p1[1])
            samples.append(((px, py), d_total))
            if k < num_steps:
                d_total += seg_len / num_steps
    if len(samples) < 2:
        return
    step = dash_len + gap_len
    in_dash = [((s[1] % step) < dash_len) for s in samples]
    for j in range(len(samples) - 1):
        if in_dash[j] and in_dash[j + 1]:
            p0 = (int(samples[j][0][0]), int(samples[j][0][1]))
            p1 = (int(samples[j + 1][0][0]), int(samples[j + 1][0][1]))
            cv2.line(img, p0, p1, color, thickness)


def compute_mask_centroid(bin_mask):
    m = cv2.moments(bin_mask.astype(np.uint8), binaryImage=True)
    if abs(m.get("m00", 0.0)) < 1e-6:
        return None
    cx = m["m10"] / m["m00"]
    cy = m["m01"] / m["m00"]
    return (float(cx), float(cy))



def draw_danger_triangle(img, anchor_xy, size=14):
    x, y = int(round(anchor_xy[0])), int(round(anchor_xy[1]))
    pts = np.array(
        [
            (x, y - size),
            (x - int(size * 0.9), y + size),
            (x + int(size * 0.9), y + size),
        ],
        dtype=np.int32,
    ).reshape(-1, 1, 2)
    cv2.fillPoly(img, [pts], DANGER_TRIANGLE_COLOR)
    cv2.polylines(img, [pts], isClosed=True, color=DANGER_TRIANGLE_OUTLINE, thickness=1)
    cv2.putText(img, "!", (x - 4, y + 6), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)



def draw_callout(img, from_xy, to_xy, text, color_arrow, color_text, bg_color, blink_on=True, prefix_danger=False):
    if from_xy is None:
        return
    p0 = (int(round(from_xy[0])), int(round(from_xy[1])))
    p1 = (int(round(to_xy[0])), int(round(to_xy[1])))

    # arrow
    cv2.arrowedLine(img, p0, p1, color_arrow, 1, tipLength=0.18)

    if not blink_on:
        return

    # text box
    font = cv2.FONT_HERSHEY_SIMPLEX
    (tw, th), baseline = cv2.getTextSize(text, font, CALLOUT_FONT_SCALE, CALLOUT_THICKNESS)

    extra_left = 0
    if prefix_danger:
        extra_left = 22

    x0 = p1[0]
    y0 = p1[1]
    rect_pt1 = (x0 - CALLOUT_PADDING, y0 - th - baseline - CALLOUT_PADDING)
    rect_pt2 = (x0 + tw + CALLOUT_PADDING + extra_left, y0 + CALLOUT_PADDING)

    cv2.rectangle(img, rect_pt1, rect_pt2, bg_color, -1)
    cv2.rectangle(img, rect_pt1, rect_pt2, (200, 200, 200), 1)

    if prefix_danger:
        tri_anchor = (x0 + 12, y0 - th // 2)
        draw_danger_triangle(img, tri_anchor, size=10)
        text_org = (x0 + extra_left, y0)
    else:
        text_org = (x0, y0)

    cv2.putText(img, text, text_org, font, CALLOUT_FONT_SCALE, color_text, CALLOUT_THICKNESS, cv2.LINE_AA)



def draw_class_callouts(overlay, mask_resized, anchor_bin, frame_idx):
    """Arrow + text for CALLOUT_CLASS_IDS; blink triangle only for BLINK_WARNING_CLASS_ID."""
    H, W = overlay.shape[:2]
    callout_offset_px = float(CALLOUT_OFFSET_CM) * float(PIXELS_PER_CM)

    def clamp_xy(xy):
        x, y = float(xy[0]), float(xy[1])
        x = max(10.0, min(W - 10.0, x))
        y = max(20.0, min(H - 20.0, y))
        return (x, y)

    blink_on = True
    if BLINK_WARNING_PERIOD_FRAMES is not None and BLINK_WARNING_PERIOD_FRAMES > 0:
        blink_on = ((frame_idx // BLINK_WARNING_PERIOD_FRAMES) % 2) == 0

    ids = list(CALLOUT_CLASS_IDS)
    n_div = max(len(ids), 1)
    for j, c in enumerate(ids):
        if c == GUIDELINE_ANCHOR_CLASS_ID:
            bin_mask = anchor_bin.astype(np.uint8)
        else:
            bin_mask = (mask_resized == c).astype(np.uint8)
        if int(bin_mask.sum()) < CALLOUT_MIN_MASK_PIXELS:
            continue
        centroid = compute_mask_centroid(bin_mask)
        if centroid is None:
            continue
        cx, cy = float(centroid[0]), float(centroid[1])
        theta = (j * 2.0 * np.pi / n_div) - (np.pi / 2.0)
        r = callout_offset_px * (1.0 + 0.2 * j)
        label_pos = clamp_xy((cx + float(np.cos(theta)) * r, cy + float(np.sin(theta)) * r))
        name = DISPLAY_CLASS_NAMES.get(c, CLASS_NAMES[c] if 0 <= c < len(CLASS_NAMES) else str(c))
        danger = BLINK_WARNING_CLASS_ID is not None and c == BLINK_WARNING_CLASS_ID
        if danger:
            draw_callout(
                overlay,
                centroid,
                label_pos,
                name,
                CALLOUT_ARROW_COLOR,
                CALLOUT_TEXT_COLOR,
                CALLOUT_BG_COLOR,
                blink_on=blink_on,
                prefix_danger=True,
            )
        else:
            draw_callout(
                overlay,
                centroid,
                label_pos,
                name,
                CALLOUT_ARROW_COLOR,
                CALLOUT_TEXT_COLOR,
                CALLOUT_BG_COLOR,
                blink_on=True,
                prefix_danger=False,
            )


def overlay_guideline(frame, mask, h, w, prev_centerline_seg=None, frame_idx=0):
    """Semi-transparent fill for OVERLAY_ALPHA_CLASS_IDS; geometry from GUIDELINE_ANCHOR_CLASS_ID."""
    overlay = frame.copy()
    mask_resized = cv2.resize(
        mask.astype(np.uint8), (frame.shape[1], frame.shape[0]),
        interpolation=cv2.INTER_NEAREST
    )
    for c in OVERLAY_ALPHA_CLASS_IDS:
        if c < 0 or c >= len(CLASS_COLORS_BGR):
            continue
        color = CLASS_COLORS_BGR[c]
        sel = mask_resized == c
        overlay[sel] = (
            (1 - ALPHA) * overlay[sel].astype(np.float32) + ALPHA * np.array(color, dtype=np.float32)
        ).astype(np.uint8)

    anchor_bin = (mask_resized == GUIDELINE_ANCHOR_CLASS_ID).astype(np.uint8)

    centerline_segments = get_pericardium_centerline(anchor_bin, gap_threshold=5, smooth_win=5)
    if not centerline_segments:
        geom_empty = {
            "main_centerline_segment_index": -1,
            "centerline_segments": [],
            "parallel_lines_offset_cm": {"positive_normal_side": [], "negative_normal_side": []},
            "band_polygons_between_parallel_bounds": [],
        }
        band_mask_empty = np.zeros((frame.shape[0], frame.shape[1]), dtype=np.uint8)
        draw_class_callouts(overlay, mask_resized, anchor_bin, frame_idx)
        return overlay.astype(np.uint8), prev_centerline_seg, geom_empty, mask_resized, band_mask_empty

    main_idx = int(np.argmax([len(s) for s in centerline_segments]))
    new_prev_centerline_seg = None

    for idx, seg in enumerate(centerline_segments):
        if idx == main_idx:
            centerline_segments[idx] = smooth_centerline_segment(seg, prev_seg_points=prev_centerline_seg)
            new_prev_centerline_seg = centerline_segments[idx]
        else:
            centerline_segments[idx] = smooth_centerline_segment(seg, prev_seg_points=None)

    offset_px = float(OFFSET_CM) * float(PIXELS_PER_CM)
    band_halfwidth_px = offset_px * (BAND_WIDTH_SCALE / 2.0)
    band_mask = np.zeros((overlay.shape[0], overlay.shape[1]), dtype=np.uint8)
    for seg in centerline_segments:
        band_poly = compute_parallel_band_polygon(seg, anchor_bin, band_halfwidth_px)
        fill_polygon_alpha(overlay, band_poly, BAND_COLOR_BGR, BAND_ALPHA)
        if band_poly is not None and len(band_poly) >= 3:
            pts = np.array(
                [(int(round(x)), int(round(y))) for x, y in band_poly], dtype=np.int32
            ).reshape(-1, 1, 2)
            cv2.fillPoly(band_mask, [pts], 255)

    for seg in centerline_segments:
        draw_dashed_polyline(overlay, seg, GUIDELINE_COLOR_BGR, LINE_THICKNESS, DASH_LEN, GAP_LEN)

    draw_class_callouts(overlay, mask_resized, anchor_bin, frame_idx)

    export_segments = [[[float(x), float(y)] for x, y in seg] for seg in centerline_segments]
    export_band_polys = []
    for seg in centerline_segments:
        poly = compute_parallel_band_polygon(seg, anchor_bin, band_halfwidth_px)
        if poly is not None:
            export_band_polys.append([[float(x), float(y)] for x, y in poly])
        else:
            export_band_polys.append([])

    if len(centerline_segments[main_idx]) >= 2:
        pos_pts, neg_pts = compute_parallel_lines(
            centerline_segments[main_idx], anchor_bin, offset_px
        )
        pos_pl = [[float(x), float(y)] for x, y in pos_pts]
        neg_pl = [[float(x), float(y)] for x, y in neg_pts]
    else:
        pos_pl, neg_pl = [], []

    geom = {
        "main_centerline_segment_index": int(main_idx),
        "centerline_segments": export_segments,
        "parallel_lines_offset_cm": {
            "positive_normal_side": pos_pl,
            "negative_normal_side": neg_pl,
        },
        "band_polygons_between_parallel_bounds": export_band_polys,
    }
    return overlay.astype(np.uint8), new_prev_centerline_seg, geom, mask_resized, band_mask

In [ ]:
all_paths = sorted(RAW_DIR.glob("frame_*.jpg"), key=lambda p: int(p.stem.split("_")[1]))
frame_paths = all_paths[:MAX_FRAMES]
print(f"Dùng {len(frame_paths)} frame đầu (tối đa {MAX_FRAMES}). Tổng trong raw: {len(all_paths)}")
if not frame_paths:
    raise SystemExit("Không có frame_*.jpg trong raw.")

In [ ]:
first = cv2.imread(str(frame_paths[0]))
if first is None:
    raise SystemExit(f"Không đọc được: {frame_paths[0]}")
h, w = first.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(str(OUTPUT_VIDEO), fourcc, FPS, (w, h))
print(f"Output video: {OUTPUT_VIDEO}, {w}x{h}, {FPS} fps")

In [ ]:
prev_centerline_seg = None

with torch.no_grad():
    for i, path in enumerate(frame_paths):
        frame = cv2.imread(str(path))
        if frame is None:
            continue
        img_t, (orig_h, orig_w) = preprocess_frame(frame)
        logits = model(img_t)
        pred = model_pred_to_semantic(
            torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        )
        _ogv = overlay_guideline(frame, pred, orig_h, orig_w, prev_centerline_seg, frame_idx=i)
        frame_overlay = _ogv[0]
        prev_centerline_seg = _ogv[1]
        out.write(frame_overlay)
        if (i + 1) % 100 == 0:
            print(f"Processed {i + 1}/{len(frame_paths)}")

out.release()
print("Done video:", OUTPUT_VIDEO)

In [ ]:
# Task 2 — export frames + band mask + binary masks from EXPORT_MASK_LAYERS (config cell).
# Run first (same kernel): config → model → definitions (preprocess … overlay_guideline).
import json
import shutil
import zipfile
from pathlib import Path

_task2_need = (
    "RAW_DIR",
    "np",
    "torch",
    "cv2",
    "model",
    "model_pred_to_semantic",
    "preprocess_frame",
    "overlay_guideline",
    "OFFSET_CM",
    "PIXELS_PER_CM",
    "BAND_WIDTH_SCALE",
    "CLASS_NAMES",
    "GUIDELINE_ANCHOR_CLASS_ID",
    "EXPORT_MASK_LAYERS",
)
_task2_missing = [n for n in _task2_need if n not in globals()]
if _task2_missing:
    raise SystemExit(
        "Task 2: run cells in order (config → model → preprocess/overlay cell). "
        f"Missing: {', '.join(_task2_missing)}"
    )

OUTPUT_SEQ_DIR = Path(r"/home/khoa/Workspace/CardioVis/Backend-Inference/features_data/")
OUTPUT_SEQ_ZIP = Path(r"/home/khoa/Workspace/CardioVis/Backend-Inference/features_data/feature_2_data.zip")

MAX_EXPORT_FRAMES = 1000
# Save class/band/foreground masks as 3-channel BGR so viewers always show white=255 (not dim single-channel).
_EXPORT_MASK_AS_BGR = True

_all_exp = sorted(RAW_DIR.glob("frame_*.jpg"), key=lambda p: int(p.stem.split("_")[1]))
export_paths = _all_exp[:MAX_EXPORT_FRAMES]
if not export_paths:
    raise SystemExit("Không có frame_*.jpg trong raw.")
print(f"Task 2: xuất {len(export_paths)} frame đầu → {OUTPUT_SEQ_DIR}")

first_e = cv2.imread(str(export_paths[0]))
if first_e is None:
    raise SystemExit(f"Không đọc được: {export_paths[0]}")
h_e, w_e = first_e.shape[:2]

shutil.rmtree(OUTPUT_SEQ_DIR, ignore_errors=True)
(OUTPUT_SEQ_DIR / "frames").mkdir(parents=True, exist_ok=True)
(OUTPUT_SEQ_DIR / "masks").mkdir(parents=True, exist_ok=True)
(OUTPUT_SEQ_DIR / "masks" / "band").mkdir(parents=True, exist_ok=True)
(OUTPUT_SEQ_DIR / "masks" / "foreground").mkdir(parents=True, exist_ok=True)
for layer in EXPORT_MASK_LAYERS:
    (OUTPUT_SEQ_DIR / "masks" / layer["subdir"]).mkdir(parents=True, exist_ok=True)
(OUTPUT_SEQ_DIR / "json").mkdir(parents=True, exist_ok=True)

_offset_px = float(OFFSET_CM) * float(PIXELS_PER_CM)
_band_half = _offset_px * (BAND_WIDTH_SCALE / 2.0)

# #region agent log
import json as _t2j, time as _t2t
_T2LOG = r"/home/khoa/Workspace/CardioVis/Backend-Inference/.cursor/debug-35efb6.log"
def _t2log(hid, msg, data):
    with open(_T2LOG, "a") as _tf:
        _tf.write(_t2j.dumps({"sessionId": "35efb6", "hypothesisId": hid, "location": "task2", "message": msg, "data": data, "timestamp": int(_t2t.time() * 1000)}) + "\n")
# #endregion

prev_centerline_seg_e = None
with torch.no_grad():
    for i, path in enumerate(export_paths):
        frame = cv2.imread(str(path))
        if frame is None:
            continue
        img_t, (orig_h, orig_w) = preprocess_frame(frame)
        logits = model(img_t)
        pred = model_pred_to_semantic(
            torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()
        )
        _og = overlay_guideline(frame, pred, orig_h, orig_w, prev_centerline_seg_e, frame_idx=i)
        if len(_og) == 5:
            _, prev_centerline_seg_e, geom, _semantic_pred, band_mask = _og
        else:
            _, prev_centerline_seg_e, geom, _semantic_pred = _og
            band_mask = np.zeros((frame.shape[0], frame.shape[1]), dtype=np.uint8)
            for poly in geom.get("band_polygons_between_parallel_bounds", []):
                if len(poly) >= 3:
                    pts = np.array(
                        [(int(round(x)), int(round(y))) for x, y in poly], dtype=np.int32
                    ).reshape(-1, 1, 2)
                    cv2.fillPoly(band_mask, [pts], 255)
        stem = f"{i:06d}"
        mr = cv2.resize(
            np.asarray(pred, dtype=np.uint8),
            (frame.shape[1], frame.shape[0]),
            interpolation=cv2.INTER_NEAREST,
        )
        if i == 0:
            _t2log("H1", "frame0_fullres_labels", {"unique_mr": [int(x) for x in np.unique(mr)], "shape": list(mr.shape)})
            print("Task2 QC frame0 unique class ids (full-res):", np.unique(mr), flush=True)
        cv2.imwrite(str(OUTPUT_SEQ_DIR / "frames" / f"frame_{stem}.png"), frame)
        _band_out = (
            cv2.cvtColor(band_mask, cv2.COLOR_GRAY2BGR)
            if _EXPORT_MASK_AS_BGR and band_mask.ndim == 2
            else band_mask
        )
        cv2.imwrite(str(OUTPUT_SEQ_DIR / "masks" / "band" / f"band_{stem}.png"), _band_out)
        _fg = ((mr > 0).astype(np.uint8) * 255)
        _fg_out = cv2.cvtColor(_fg, cv2.COLOR_GRAY2BGR) if _EXPORT_MASK_AS_BGR else _fg
        cv2.imwrite(str(OUTPUT_SEQ_DIR / "masks" / "foreground" / f"foreground_{stem}.png"), _fg_out)
        mask_files = {
            "band_guideline_between_parallel": f"masks/band/band_{stem}.png",
            "foreground_any_non_background": f"masks/foreground/foreground_{stem}.png",
        }
        mask_notes = {
            "band_guideline_between_parallel": (
                "0/255 band region (guideline), not full anchor-class mask."
            ),
            "foreground_any_non_background": (
                "0/255 union of all predicted classes (mr>0); BGR PNG if _EXPORT_MASK_AS_BGR."
            ),
        }
        for layer in EXPORT_MASK_LAYERS:
            cid = int(layer["class_id"])
            sub = layer["subdir"]
            pfx = layer["file_prefix"]
            jk = layer["json_key"]
            note = layer.get("note", "")
            bin_m = ((mr == cid).astype(np.uint8) * 255)
            if i == 0:
                _t2log("H2", "frame0_class_layer", {"class_id": cid, "fg_pixels_255": int(bin_m.sum() // 255)})
            _bin_out = cv2.cvtColor(bin_m, cv2.COLOR_GRAY2BGR) if _EXPORT_MASK_AS_BGR else bin_m
            fn = f"{pfx}{stem}.png"
            cv2.imwrite(str(OUTPUT_SEQ_DIR / "masks" / sub / fn), _bin_out)
            mask_files[jk] = f"masks/{sub}/{fn}"
            mask_notes[jk] = note
        payload = {
            "frame_index": i,
            "source_path": str(path),
            "task": "export_500",
            "frame_file": f"frames/frame_{stem}.png",
            "frame_note": "raw BGR, không overlay",
            "predict_note": "Single model.forward; class masks from argmax (no second inference).",
            "mask_files": mask_files,
            "mask_notes": mask_notes,
            "image_size": {"width": int(w_e), "height": int(h_e)},
            "class_names": {str(k): v for k, v in enumerate(CLASS_NAMES)},
            "pericardium_class": int(GUIDELINE_ANCHOR_CLASS_ID),
            "params": {
                "OFFSET_CM": float(OFFSET_CM),
                "PIXELS_PER_CM": float(PIXELS_PER_CM),
                "offset_px": float(_offset_px),
                "BAND_WIDTH_SCALE": float(BAND_WIDTH_SCALE),
                "band_halfwidth_px": float(_band_half),
            },
            **geom,
        }
        with open(OUTPUT_SEQ_DIR / "json" / f"ann_{stem}.json", "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)
        if (i + 1) % 100 == 0:
            print(f"Task 2: {i + 1}/{len(export_paths)}")

if OUTPUT_SEQ_ZIP.exists():
    OUTPUT_SEQ_ZIP.unlink()
with zipfile.ZipFile(OUTPUT_SEQ_ZIP, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in OUTPUT_SEQ_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=p.relative_to(OUTPUT_SEQ_DIR))

print("Task 2 done. Folder:", OUTPUT_SEQ_DIR)
print("ZIP:", OUTPUT_SEQ_ZIP)
